# Pareto front generation via the augmented ε-constraint method

This example traces the **energy-vs-comfort Pareto front** of a thermal zone model with
`Optimizer.pareto_front` — no evolutionary algorithm involved. Because the Twin4Build
simulator is differentiable, every front point is an *exact gradient-based NLP solve*:

1. **Anchor solves**: minimize each objective alone → ideal/nadir estimates of the second objective.
2. **ε sweep**: keep f₁ (heater energy, `min`) as the objective, demote f₂ (indoor temperature,
   `max`) to a hard constraint `f2_norm ≤ ε`, and sweep ε between the anchors. A small
   `δ·f2_norm` objective term (the AUGMECON augmentation) guarantees *properly* Pareto-optimal
   points, and — unlike a weighted sum — the ε-constraint scheme recovers non-convex front regions.
3. **Batched prepass** (default on): all ε-subproblems are first solved approximately as ONE
   batched torch loss (`torch.func.vmap` over the composed rollout — one backward pass per
   iteration yields every copy's gradient), then the exact sequential SLSQP solves merely polish
   the batched solutions. This is exactly the batched workload shape where a GPU pays off:
   on a CUDA machine, add `model.to("cuda")` after `model.load()`.

As a bonus, the front slope `-df₁/dε` is reported per point: the *marginal energy price of
comfort* along the front.


In [ ]:
# Install (uncomment on Colab)
# %pip install -q twin4build

import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dateutil import tz

import twin4build as tb

# One thermal zone: outdoor at 5 degC, a controllable heater (0-3000 W), and a
# temperature sensor. The heater schedule trajectory (one value per hour over
# 24 h) is the decision variable.
START = datetime.datetime(2024, 1, 4, tzinfo=tz.gettz("Europe/Copenhagen"))
END = START + datetime.timedelta(hours=24)
STEP = 3600

model = tb.Model(id="pareto_example")
zone = tb.BuildingSpaceThermalTorchSystem(
    C_air=1e6, C_wall=5e6, R_out=0.01, R_in=0.01,
    f_wall=0.0, f_air=0.0, Q_occ_gain=100.0, id="Zone",
)
outdoor = tb.ScheduleSystem(weekDayRulesetDict={"ruleset_default_value": 5.0}, id="Outdoor")
zero = tb.ScheduleSystem(weekDayRulesetDict={"ruleset_default_value": 0.0}, id="Zero")
supply = tb.ScheduleSystem(weekDayRulesetDict={"ruleset_default_value": 20.0}, id="SupplyAirTemp")
heater = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 0.0,
        "ruleset_start_minute": [0], "ruleset_end_minute": [0],
        "ruleset_start_hour": [6], "ruleset_end_hour": [20],
        "ruleset_value": [1500.0],
    },
    id="Heater",
)

model.add_connection(outdoor, zone, "scheduleValue", "outdoorTemperature")
model.add_connection(zero, zone, "scheduleValue", "supplyAirFlowRate")
model.add_connection(zero, zone, "scheduleValue", "exhaustAirFlowRate")
model.add_connection(supply, zone, "scheduleValue", "supplyAirTemperature")
model.add_connection(zero, zone, "scheduleValue", "globalIrradiation")
model.add_connection(zero, zone, "scheduleValue", "numberOfPeople")
model.add_connection(heater, zone, "scheduleValue", "heatGain")
model.load(draw_semantic_model=False, draw_simulation_model=False)

# GPU: uncomment on a CUDA machine -- the batched prepass is the part that benefits.
# model.to("cuda")

simulator = tb.Simulator(model)
optimizer = tb.Optimizer(simulator)


## Trace the front

`objective1` is kept as the scalar objective; `objective2` is swept via the ε constraint.
Both use the familiar `(component, port, "min"|"max")` format from `Optimizer.optimize`.


In [ ]:
result = optimizer.pareto_front(
    start_time=START,
    end_time=END,
    step_size=STEP,
    variables=[(heater, "scheduleValue", 0.0, 3000.0)],
    objective1=(heater, "scheduleValue", "min"),        # mean heater power [W]
    objective2=(zone, "indoorTemperature", "max"),      # mean indoor temperature [degC]
    n_points=8,
    batched_prepass=True,
    options={"maxiter": 50},
)

df = pd.DataFrame({
    "eps": result.eps,
    "mean heater power [W]": result.f1,
    "mean indoor temp [degC]": result.f2,
    "slope -df1/deps": result.slope,
    "success": result.success,
    "iterations": result.nit,
    "pareto": result.pareto_mask,
})
df


## The front

Each point is a full 24-h control trajectory. The slope annotation is the local exchange
rate: how much extra (normalized) energy one more unit of comfort costs at that point.


In [ ]:
ax = result.plot()
for i in range(1, len(result.eps) - 1):
    ax.annotate(
        f"{result.slope[i]:.2f}",
        (result.f2[i], result.f1[i]),
        textcoords="offset points", xytext=(8, -4), fontsize=8, alpha=0.8,
    )
ax.set_title("Energy vs comfort Pareto front (AUGMECON)")
plt.tight_layout()
plt.show()


## Inspect one solution

`result.apply(i)` writes point `i`'s decision trajectories back into the model and
re-simulates, so the component histories hold that solution.


In [ ]:
i = len(result.eps) // 2  # a mid-front compromise
result.apply(i)

t = np.arange(24)
power = heater.output["scheduleValue"].history(i_s=0).detach().cpu().numpy().ravel()
temp = zone.output["indoorTemperature"].history(i_s=0).detach().cpu().numpy().ravel()

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.step(t, power, where="post", color="tab:red", label="heater power")
ax1.set_xlabel("hour")
ax1.set_ylabel("heater power [W]", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(t, temp, color="tab:blue", label="indoor temperature")
ax2.set_ylabel("indoor temperature [degC]", color="tab:blue")
ax1.set_title(
    f"Pareto point {i}: mean power {result.f1[i]:.0f} W, "
    f"mean temp {result.f2[i]:.1f} degC"
)
plt.tight_layout()
plt.show()


## Notes and limits

- **GPU batching**: the sequential SLSQP polish is CPU-bound (scipy), but the batched
  prepass evaluates all ε-copies as one tensor program — with `model.to("cuda")` that is
  the batched-kernel workload where the GPU pays off (see the GPU scaling benchmark
  notebook). The prepass typically leaves each subproblem a few SLSQP iterations from
  optimality.
- **Bi-objective only**: ε-grids scale poorly beyond ~3 objectives (use NBI-style methods
  there). A uniform ε grid gives non-uniform point spacing on steep front segments.
- **Local optimality**: each point inherits the NLP's non-convexity; the dominance filter
  (`result.pareto_mask`) removes any point that ends up dominated.
- The reported slope is a finite-difference estimate; exact constraint multipliers arrive
  with the IPOPT-based batched finisher (follow-up work).
